# PersuasiX — Evaluation & Analysis

Comprehensive evaluation of all four tasks with visualizations.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.metrics import compute_detection_metrics, compute_generation_metrics
from src.utils.visualization import plot_confusion_matrix
from src.data.collector import TECHNIQUE_LABELS

## 1. Detection Evaluation (Simulated)

In [ ]:
# Simulate predictions for demonstration
np.random.seed(42)
n_samples = 100
n_labels = len(TECHNIQUE_LABELS)

y_true = (np.random.rand(n_samples, n_labels) > 0.85).astype(int)
y_pred = y_true.copy()
# Flip ~10% of predictions to simulate imperfect model
flip_mask = np.random.rand(n_samples, n_labels) < 0.1
y_pred[flip_mask] = 1 - y_pred[flip_mask]

metrics = compute_detection_metrics(y_true, y_pred, TECHNIQUE_LABELS)
print(f"F1 Macro: {metrics['f1_macro']:.4f}")
print(f"F1 Micro: {metrics['f1_micro']:.4f}")
print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")
print(f"Exact Match: {metrics['exact_match_ratio']:.4f}")

In [ ]:
# Per-technique F1 scores
per_tech = metrics['per_technique']
tech_f1 = {k: v['f1'] for k, v in per_tech.items()}

fig, ax = plt.subplots(figsize=(12, 6))
sorted_items = sorted(tech_f1.items(), key=lambda x: x[1], reverse=True)
names = [k.replace('_', ' ').title() for k, _ in sorted_items]
scores = [v for _, v in sorted_items]

colors = ['#2ecc71' if s > 0.7 else '#f39c12' if s > 0.5 else '#e74c3c' for s in scores]
ax.barh(names, scores, color=colors)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Technique F1 Scores')
ax.axvline(x=0.7, color='gray', linestyle='--', alpha=0.5, label='Threshold (0.7)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig = plot_confusion_matrix(y_true, y_pred, TECHNIQUE_LABELS)
plt.show()

## 2. Generation Metrics (Simulated)

In [ ]:
# Simulate explanation predictions
references = [
    'This text uses appeal to fear by painting a catastrophic future without evidence.',
    'The text employs loaded language and name-calling to discredit the opponent.',
    'This uses bandwagon fallacy by claiming everyone agrees with the position.',
]
predictions = [
    'This text uses fear tactics by describing a catastrophic scenario without supporting evidence.',
    'The text uses emotionally charged language and personal attacks against the opponent.',
    'This text claims universal agreement to pressure the reader into conforming.',
]

gen_metrics = compute_generation_metrics(predictions, references)
for k, v in gen_metrics.items():
    print(f'{k:25s}: {v:.4f}')

## 3. Cross-Lingual Analysis

In [ ]:
# Test cross-lingual consistency with the scorer
try:
    from src.models.scorer import ReliabilityScorer
    
    scorer = ReliabilityScorer(device='cpu')
    
    result = scorer.cross_lingual_similarity({
        'en': 'This text uses appeal to fear by painting a catastrophic future.',
        'fr': "Ce texte utilise l'appel à la peur en dépeignant un avenir catastrophique.",
        'ar': 'يستخدم هذا النص التخويف برسم مستقبل كارثي.',
    })
    
    print('Cross-lingual similarity:')
    for pair, sim in result.items():
        print(f'  {pair}: {sim}')
except Exception as e:
    print(f'Scorer not available: {e}')